In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pprint

from generate_simple_panel import generate_panel
from make_split import SplitGenerator

In [ ]:
panel = generate_panel()

In [ ]:
print(panel.columns)
print()
print(panel.shape)
print()
print(panel.head())

In [ ]:
n_treated = 0
n_control = 0
n_cohort = {}

for group, data in panel.groupby("firm_id"):
    is_treated = int(data["treated"].iloc[0])
    is_control = int(data["control"].iloc[0])
    cohort = int(data["cohort"].iloc[0])

    n_treated += is_treated
    n_control += is_control

    if cohort >= 0:
        if cohort not in n_cohort:
            n_cohort[cohort] = {}
        n_cohort[cohort]["treated"] = n_cohort[cohort].get("treated", 0) + is_treated
        n_cohort[cohort]["control"] = n_cohort[cohort].get("control", 0) + is_control

print(f"Number of treated firms: {n_treated}")
print(f"Number of control firms: {n_control}")
print(f"Number of cohorts: {len(n_cohort)}")
print("Cohort sizes:")
pprint.pprint(n_cohort)

In [ ]:
split_generator = SplitGenerator(panel, train_nini_ratio=0.5, seed=42)
split = split_generator.generate()
print(split)

In [ ]:
train = split['train']
test = split['test']

treated = train['T']
nini_train = train['NiNi']

control = test['C']
nini_test = test['NiNi']

## Chequeo de seguridad
for id in treated:
    firm = panel.groupby("firm_id").get_group(id)
    assert firm["treated"].iloc[0] == True, f"Firm {id} is not treated"
    assert firm["control"].iloc[0] == False, f"Firm {id} is control"

for id in control:
    firm = panel.groupby("firm_id").get_group(id)
    assert firm["control"].iloc[0] == True, f"Firm {id} is not control"
    assert firm["treated"].iloc[0] == False, f"Firm {id} is not treated"

for id in nini_train + nini_test:
    firm = panel.groupby("firm_id").get_group(id)
    assert firm["treated"].iloc[0] == False and firm["control"].iloc[0] == False, f"Firm {id} is not NiNi"